# Python Package Usuage

## OpenStreetMap API Using `requests` Package

### OpenStreetMap API Documentation

[Click Here](https://nominatim.org/release-docs/latest/)


In [ ]:
# Install the requests package using uv or pip

#  uncomment the line below to use pip instead of uv
# ! pip install
# ! pip install pandas
# !pip install geopy
# ! pip install openpyxl
! uv add requests
! uv add pandas
! uv add geopy
! uv add openpyxl

In [ ]:
# imports
import requests
import pandas as pd
from geopy.distance import geodesic

from typing import List, Dict, Any
from requests import Response
from requests.exceptions import RequestException

### Search Places -> Format Result -> Find Places Within The Radius

In [ ]:
def search_places(
    lat: float,
    lon: float,
    query: str,
    radius_km: float
) -> List[Dict[str, Any]]:
    """Search for nearby places using Nominatim API.

    Args:
        lat (float): Latitude of the location.
        lon (float): Longitude of the location.
        query (str): Search query (e.g., "restaurant").
        radius_km (float): Search radius in kilometers.

    Returns:
        List[Dict[str, Any]]: List of places with their details.
    """
    # Convert radius to bounding box (approx)
    delta: float = radius_km / 111  
    viewbox: str = f"{lon-delta},{lat+delta},{lon+delta},{lat-delta}"

    url: str = "https://nominatim.openstreetmap.org/search"

    params: Dict[str, Any] = {
        "q": query,
        "format": "json",
        "limit": 50,
        "viewbox": viewbox,
        "bounded": 1
    }

    headers: Dict[str, str] = {
        "User-Agent": "Mozilla/5.0 (NearbyPlacesApp)"
    }

    try:
        response: Response = requests.get(url, params=params, headers=headers)
        return response.json()
    except RequestException as e:
        print(f"Error fetching data: {e}")
        return [{}]
    except Exception as e:
        print(f"Unexpected error: {e}")
        return [{}]

def format_results(
    data: List[Dict[str, Any]],
    lat: float,
    lon: float,
    radius_km: float
) -> List[Dict[str, Any]]:
    """Analyze and format the raw results from Nominatim API.
    
    Args:
        data: List of dictionaries containing raw place data from Nominatim API.
        lat: Latitude of the reference point.
        lon: Longitude of the reference point.
        radius_km: Radius in kilometers to filter places.

    Returns:
        A list of dictionaries containing formatted place information within the specified radius.
    """
    results: List[Dict[str, Any]] = []
    for place in data:
        place_lat = float(place["lat"])
        place_lon = float(place["lon"])
        distance = geodesic((lat, lon), (place_lat, place_lon)).km

        if distance <= radius_km:
            results.append({
                "Name": place.get("display_name", "Unknown"),
                "Latitude": place_lat,
                "Longitude": place_lon,
                "Distance (km)": round(distance, 2)
            })

    return results

def export_results(
    results: List[Dict[str, Any]],
    filename: str ="nominatim_places.xlsx"
) -> None:

    df = pd.DataFrame(results)
    if df.empty:
        print("No results found.")
        return

    df = df.sort_values("Distance (km)")
    print(df)
    df.to_excel(filename, index=False)
    print(f"Saved to {filename}")

### Take Input From user

In [ ]:
# Get information from user

# search_places(43.69790866541507, -79.77607788294328, query="bank", radius_km=3)
try:
    latitude = float(input("Enter Latitude (must be a valid numeric value):  "))
    longitude = float(input("Enter Longitude (must be a valid numeric value):  "))
    
    query = input("Enter search query (e.g., 'restaurant', 'bank', 'cafe', 'hospital', 'school'), default: 'restaurant': ")
    if not query:
        query = "restaurant"
    else:
        query = query.strip().lower()
        if query.isnumeric():
            raise ValueError("Query should be a non-numeric string.")
        elif query not in ["restaurant", "bank", "cafe", "hospital", "school"]:
            raise ValueError("Query must be one of the predefined options: 'restaurant', 'bank', 'cafe', 'hospital', 'school'.")
    
    radius_km = input("Enter search radius in km (must be a valid numeric value, default 3):  ")
    if not radius_km:
        radius_km = 3
    else:
        radius_km = float(radius_km)
except ValueError as e:
    print(f"Invalid input: {e}")

### Call Search Places

In [ ]:
places: List[Dict[str, Any]] = search_places(lat=latitude, lon=longitude, query=query, radius_km=radius_km)

### Reformat Result

In [ ]:
formatted_result: List[Dict[str, Any]] = format_results(data=places, lat=latitude, lon=longitude, radius_km=radius_km)

### Generate Excel

In [ ]:
export_results(results=formatted_result, filename="nominatim_places.xlsx")